In [9]:
import requests
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):

    url = f"https://rest.uniprot.org/uniprotkb/{accession}"
    params = {'format': 'json'}
    return requests.get(url, params=params)


def uniprot_parse_response(resp):

    if not resp.ok:
        return None

    data = resp.json()
    try:
        parsed_data = {
            'organism': data.get('organism', {}).get('scientificName'),
            'geneInfo': data.get('genes', []),
            'sequenceInfo': data.get('sequence', {}),
            'type': 'protein'
        }
        return {data.get('primaryAccession'): parsed_data}
    except Exception:
        return None

def get_ensembl(ensembl_id):

    url = f"https://rest.ensembl.org/lookup/id/{ensembl_id}"
    params = {"content-type": "application/json"}
    return requests.get(url, params=params)

def ensembl_parse_response(resp):

    if not resp.ok:
        return None

    data = resp.json()
    keys = [
        'object_type', 'assembly_name', 'species', 'db_type', 'biotype',
        'display_name', 'id', 'description', 'canonical_transcript', 'source'
    ]
    parsed_info = {key: data.get(key) for key in keys}
    return {data.get('id'): parsed_info}


def main(ids: list):

    uniprot_re = r"^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
    ensembl_re = r"^ENS[A-Z]*[GTP][0-9]{11}$"

    all_results = {}

    for entry_id in ids:
        if re.match(uniprot_re, entry_id):
            resp = get_uniprot(entry_id)
            parsed = uniprot_parse_response(resp)
            if parsed:
                all_results.update(parsed)
            else:
                all_results[entry_id] = {"error": "ID not found"}
        elif re.match(ensembl_re, entry_id):
            resp = get_ensembl(entry_id)
            parsed = ensembl_parse_response(resp)
            if parsed:
                all_results.update(parsed)
            else:
                all_results[entry_id] = {"error": "ID not found"}
        else:
            all_results[entry_id] = {"error": "unknown database"}

    return pd.DataFrame.from_dict(all_results, orient='index')

if __name__ == "__main__":
    test_data = ['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618']
    result_df = main(test_data)

    print(result_df)

                                   organism  \
P11473                         Homo sapiens   
Q91XI3           Ictidomys tridecemlineatus   
hello                                   NaN   
ENSG00000157764                         NaN   
ENSG00000139618                         NaN   

                                                          geneInfo  \
P11473           [{'geneName': {'evidences': [{'evidenceCode': ...   
Q91XI3                            [{'geneName': {'value': 'INS'}}]   
hello                                                          NaN   
ENSG00000157764                                                NaN   
ENSG00000139618                                                NaN   

                                                      sequenceInfo     type  \
P11473           {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...  protein   
Q91XI3           {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...  protein   
hello                                                      